## 1. Install & Import Libraries

In [ ]:
# Uncomment to install if needed:
# !pip install requests pandas tqdm chembl-webresource-client

import requests
import pandas as pd
import numpy as np
import time
import os
from tqdm.auto import tqdm

print("Libraries loaded successfully.")

## 2. Configuration

In [ ]:
# ── Target Settings ──────────────────────────────────────────────
TARGET_CHEMBL_ID = "CHEMBL1741208"   # Human NLRP3 inflammasome
TARGET_NAME      = "NLRP3 (Homo sapiens)"

# ── Output paths ─────────────────────────────────────────────────
os.makedirs("../data/raw", exist_ok=True)

OUT_ACTIVITIES = "../data/raw/chembl_nlrp3_activities_raw.csv"
OUT_MOLECULES  = "../data/raw/chembl_nlrp3_unique_molecules.csv"
OUT_MERGED     = "../data/raw/chembl_nlrp3_merged.csv"
OUT_IC50       = "../data/raw/chembl_nlrp3_ic50_unique.csv"

# ── API base ─────────────────────────────────────────────────────
CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"
PAGE_SIZE   = 100
SLEEP_SEC   = 0.15   # polite delay between requests

print(f"Target : {TARGET_NAME}")
print(f"ChEMBL ID: {TARGET_CHEMBL_ID}")

## 3. Download All Activity Records

In [ ]:
def fetch_activities(target_id: str, page_size: int = 100) -> pd.DataFrame:
    """Page through ChEMBL activity endpoint and return a DataFrame."""
    url    = f"{CHEMBL_BASE}/activity.json"
    params = {"target_chembl_id": target_id, "limit": page_size, "offset": 0}
    rows   = []

    print(f"Downloading activities for {target_id} …")
    while True:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        acts = data.get("activities", [])
        if not acts:
            break
        for a in acts:
            rows.append({
                "activity_chembl_id"  : a.get("activity_chembl_id"),
                "molecule_chembl_id"  : a.get("molecule_chembl_id"),
                "assay_chembl_id"     : a.get("assay_chembl_id"),
                "standard_type"       : a.get("standard_type"),
                "standard_relation"   : a.get("standard_relation"),
                "standard_value"      : a.get("standard_value"),
                "standard_units"      : a.get("standard_units"),
                "pchembl_value"       : a.get("pchembl_value"),
                "data_validity_comment": a.get("data_validity_comment"),
                "document_year"       : a.get("document_year"),
            })
        # Pagination
        next_url = data.get("page_meta", {}).get("next")
        if not next_url:
            break
        params["offset"] += page_size
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(rows)
    print(f"  → {len(df):,} activity records downloaded.")
    return df

act_df = fetch_activities(TARGET_CHEMBL_ID, PAGE_SIZE)
act_df.to_csv(OUT_ACTIVITIES, index=False)
print(f"Saved → {OUT_ACTIVITIES}")

## 4. Fetch Molecule Properties for Unique Compounds

In [ ]:
def fetch_molecule_info(chembl_id: str) -> dict:
    """Retrieve molecular properties for a single ChEMBL compound."""
    url  = f"{CHEMBL_BASE}/molecule/{chembl_id}.json"
    resp = requests.get(url, timeout=30)
    if resp.status_code != 200:
        return {}
    m = resp.json()
    props = m.get("molecule_properties") or {}
    structs = m.get("molecule_structures") or {}
    return {
        "molecule_chembl_id"   : chembl_id,
        "pref_name"            : m.get("pref_name"),
        "max_phase"            : m.get("max_phase"),
        "molecule_type"        : m.get("molecule_type"),
        "canonical_smiles"     : structs.get("canonical_smiles"),
        "standard_inchi_key"   : structs.get("standard_inchi_key"),
        "molecular_weight"     : props.get("full_mwt"),
        "alogp"                : props.get("alogp"),
        "ro5_violations"       : props.get("num_ro5_violations"),
        "hbd"                  : props.get("hbd"),
        "hba"                  : props.get("hba"),
        "psa"                  : props.get("psa"),
        "rtb"                  : props.get("rtb"),
        "chembl_url"           : f"https://www.ebi.ac.uk/chembl/compound_report_card/{chembl_id}",
    }

unique_mols = act_df["molecule_chembl_id"].dropna().unique().tolist()
print(f"Unique molecules to enrich: {len(unique_mols):,}")

mol_rows = []
for mid in tqdm(unique_mols, desc="Fetching molecule info"):
    info = fetch_molecule_info(mid)
    if info:
        mol_rows.append(info)
    time.sleep(SLEEP_SEC)

mol_df = pd.DataFrame(mol_rows)
mol_df.to_csv(OUT_MOLECULES, index=False)
print(f"Saved → {OUT_MOLECULES}  ({len(mol_df):,} molecules)")

## 5. Merge Activities + Molecules

In [ ]:
merged = act_df.merge(mol_df, on="molecule_chembl_id", how="left")
merged.to_csv(OUT_MERGED, index=False)
print(f"Merged dataset shape : {merged.shape}")
print(f"Saved → {OUT_MERGED}")
merged.head(3)

## 6. Extract IC50-Only Unique Molecules

In [ ]:
ic50_df = merged[merged["standard_type"] == "IC50"].copy()
ic50_df["standard_value"] = pd.to_numeric(ic50_df["standard_value"], errors="coerce")
ic50_df = ic50_df.dropna(subset=["standard_value", "canonical_smiles"])
ic50_df = ic50_df.drop_duplicates(subset=["standard_inchi_key"])

ic50_df.to_csv(OUT_IC50, index=False)

print(f"IC50 unique molecules : {len(ic50_df):,}")
print(f"Saved → {OUT_IC50}")
ic50_df.head(3)

## 7. Summary

In [ ]:
print("=" * 55)
print("  DATA EXTRACTION SUMMARY")
print("=" * 55)
print(f"  Target            : {TARGET_NAME}")
print(f"  ChEMBL Target ID  : {TARGET_CHEMBL_ID}")
print(f"  Total activities  : {len(act_df):,}")
print(f"  Unique molecules  : {len(mol_df):,}")
print(f"  IC50 unique mols  : {len(ic50_df):,}")
print("=" * 55)
print("Activity type breakdown:")
print(act_df["standard_type"].value_counts().head(10).to_string())